## Settings

In [ ]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [ ]:
## libraries
import sys
import logging
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.estimators.factories import load_estimators
from src.data.builders import (
    load_processed_data,
    load_perturbed_data
)
from src.evaluators.perturbing import (
    train_perturbed_transfer,
    train_perturbed_recovery,
    train_perturbed_consensus,
    compile_perturbed_transfer,
    compile_perturbed_recovery,
    compile_perturbed_consensus,
    find_perturbed_max,
    spec_marginal_delta,
    stat_perturbed_tost
)
from src.visualizers.visualizing import plot_consensus

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z,
    TARGET
)

## Initialization

In [ ]:
## reproducibility
N_REPEATS = 30
RANDOM_STATE = 42

## load data and models
_disable = logging.root.manager.disable
logging.disable(logging.INFO)
try:
    data_proc = load_processed_data()
    data_pert_all = load_perturbed_data()
    data_pert = {
        key: data_pert_all[key]
        for key in [
            "network_perturbed", 
            "invariants_perturbed", 
            "process_perturbed", 
            "signatures_perturbed"
        ]
        if key in data_pert_all
    }
finally:
    logging.disable(_disable)
    models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data_proc.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")
print("Perturbed Data:")
for json_key, methods in data_pert.items():
    print(f"  {json_key}:")
    for method, intense in methods.items():
        pert_df = next(iter(intense.values()))
        n_r, n_c = pert_df.shape
        print(f"   - {method}: {n_c} features, {n_r} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## Training

In [ ]:
## perturbation transfer training
if "results_dict_perturbed_transfer" not in globals():
    results_dict_perturbed_transfer = train_perturbed_transfer(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
    )

In [ ]:
## perturbation recovery training
if "results_dict_perturbed_recovery" not in globals():
    results_dict_perturbed_recovery = train_perturbed_recovery(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )

In [ ]:
## perturbation pairwise consensus training
if "results_dict_perturbed_consensus" not in globals():
    results_dict_perturbed_consensus = train_perturbed_consensus(
        data = data_proc,
        models = models,
        data_pert = data_pert,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        target = TARGET,
        n_repeats = N_REPEATS,
        random_state = RANDOM_STATE,
    )

## Post-Processing

In [ ]:
## compile perturbation transfer results
results_perturbed_transfer, recovery_perturbed_transfer = compile_perturbed_transfer(
    results = results_dict_perturbed_transfer
)

## compile perturbation structural recovery results
results_perturbed_recovery = compile_perturbed_recovery(
    results = results_dict_perturbed_recovery
)

## compile perturbation pairwise consensus results
results_perturbed_consensus = compile_perturbed_consensus(
    results = results_dict_perturbed_consensus
)

## Perturbation Transfer Equivalence Test
Each perturbation method is tested for equivalence against the unperturbed baseline EI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed EI values are paired on (model $\times$ domain), matching the falsification pipeline's pairing convention. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners on the original EI transfer results. Half the interquartile range of the baseline EI values sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## maximum intensity per perturbation method
results_perturbed_transfer_max = find_perturbed_max(
    results = results_perturbed_transfer
)

## empirical delta specification
delta_transfer = spec_marginal_delta(
    results = results_perturbed_transfer,
    feat_value = ["ei"],
    track = "frozen",
    method = "iqr",
    scale = 1.0,  ## iqr range
    decimals = 3
)

## tost equivalence test across all perturbation methods
results_transfer = stat_perturbed_tost(
    results = results_perturbed_transfer_max,
    feat_value = ["ei"],
    feat_group = ["perturbation", "method"],
    track = "frozen",
    delta = delta_transfer,
    decimals = 3
)

display(results_transfer)

All four perturbation methods achieve TOST equivalence at maximum intensity. Effect sizes are negligible across all methods. The largest median shift (0.006, process) consumes less than one-quarter of the equivalence margin. EI is robust to aggressive corruption of network structure and its graph invariant vector representation, as well as stochastic process and its process signature vector representation.

### Structural Recovery Evaluation
Whereas the EI transfer test above probes whether perturbations distort transfer performance, the structural recovery test probes whether predictions preserve the observed capacity ordering. Predictions are generated under the same frozen LOGO-CV (domain) protocol used for EI, then scored by the composite consensus index CI = f(rho, rbo, dcr) per (model x domain).

## Structural Recovery Equivalence Test
Each perturbation method is tested for statistical equivalence against the unperturbed baseline CI using two one-sided tests (TOST) with paired Wilcoxon signed-rank statistics.

- **$H_0$**: The absolute difference meets or exceeds the margin ($|\Delta| \ge \delta$).
- **$H_1$**: The absolute difference falls strictly within the margin ($|\Delta| < \delta$).

Rejecting $H_0$ establishes that the perturbation effect is negligibly small relative to $\delta$.

### Maximum-Intensity Protocol
Only the highest intensity setting per perturbation method enters the test. Baseline and perturbed CI values are paired on (model $\times$ domain), matching the transfer EI test. This yields one paired comparison per model $\times$ domain $\times$ method and prevents a dense intensity sweep from inflating sample size.

### Empirical Test Specification
The equivalence margin $\delta$ is derived from baseline variability across learners on the original structural recovery CI results. Half the interquartile range of the baseline CI values sets $\delta$, anchoring the margin to natural performance dispersion rather than to perturbation effects.

In [ ]:
## structural recovery equivalence at maximum perturbation intensity
results_perturbed_recovery_max = find_perturbed_max(
    results = results_perturbed_recovery
)

delta_recovery = spec_marginal_delta(
    results = results_perturbed_recovery,
    metric = "ci",
    baseline_name = "baseline",
    scale = 0.5,
)

results_recovery = stat_perturbed_tost(
    results = results_perturbed_recovery_max,
    metric = "ci",
    baseline_name = "baseline",
    alpha = 0.05,
    delta = delta_recovery,
)
display(results_recovery)


## Pairwise Consensus Equivalence Test
The CI test above probes recovery between each model and the observed target; the pairwise consensus test probes agreement between models. Two models can each recover the target while drifting apart from one another, so this is a stricter test of perturbation neutrality.

Pairwise agreement is measured with the same CI composite, but now comparing model prediction vectors within each domain. Baseline variability again sets the equivalence margin $\delta$, and only maximum perturbation intensity is tested.

In [ ]:
## paradigm-level pairwise consensus heatmaps
results_perturbed_consensus_max_current = globals().get(
    "results_perturbed_consensus_max"
)
if results_perturbed_consensus_max_current is None:
    results_perturbed_consensus_max_current = find_perturbed_max(
        results = results_perturbed_consensus,
    )

results_perturbed_consensus_max = results_perturbed_consensus_max_current
results_perturbed_consensus_plot = results_perturbed_consensus_max.loc[
    results_perturbed_consensus_max["perturbation"] != "baseline"
].copy()

fig, axes, perturbed_consensus_matrices = plot_consensus(
    results = results_perturbed_consensus_plot,
    title = "Perturbed Cross-Model Consensus by Learning Paradigm"
)